# Data Preprocessing 

## Converting height and weight to standard measurements 


In [2]:
# Imports and data labeling.
import pandas as pd
from pathlib import Path

project_dir = Path.cwd()
if project_dir.name == "src":
    project_dir = project_dir.parent
elif project_dir.name != "Frailty":
    project_dir = project_dir / "Frailty"

raw_data_path = project_dir / "data_raw" / "Frality_Raw_Data.csv"
data = pd.read_csv(raw_data_path).rename(columns={
    "Height": "Height_in",
    "Weight": "Weight_lb",
    "Age": "Age_yr",
    "Grip strength": "Grip_kg",
})

data.head()

,Height_in,Weight_lb,Age_yr,Grip_kg,Frailty
0,65.8,112,30,30,N
1,71.5,136,19,31,N
2,69.4,153,45,29,N
3,68.2,142,22,28,Y
4,67.8,144,29,24,Y


In [3]:
# Standardize units, engineer features, and encode categorical values.
data["Height_m"] = data["Height_in"] * 0.0254
data["Weight_kg"] = data["Weight_lb"] * 0.45359237
data["BMI"] = (data["Weight_kg"] / (data["Height_m"] ** 2)).round(2)

data["AgeGroup"] = pd.cut(
    data["Age_yr"],
    bins=[float("-inf"), 30, 46, 61, float("inf")],
    labels=["<30", "30–45", "46–60", ">60"],
    right=False,
)
data["Frailty_binary"] = data["Frailty"].map({"Y": 1, "N": 0}).astype("int8")

age_group_columns = [
    "AgeGroup_<30",
    "AgeGroup_30–45",
    "AgeGroup_46–60",
    "AgeGroup_>60",
]
age_group_encoded = pd.get_dummies(data["AgeGroup"], prefix="AgeGroup", dtype="int8")
data = pd.concat([data, age_group_encoded], axis=1)
for column in age_group_columns:
    if column not in data:
        data[column] = 0

data[age_group_columns] = data[age_group_columns].astype("int8")

clean_dir = project_dir / "data_clean"
clean_dir.mkdir(exist_ok=True)
clean_data_path = clean_dir / "frailty_processed.csv"
data.to_csv(clean_data_path, index=False)

print(f"Saved processed data to {clean_data_path}")
data

Saved processed data to /Users/randbrown/Desktop/Principle of Data Science/Assignment 1/Frailty/data_clean/frailty_processed.csv


,Height_in,Weight_lb,Age_yr,Grip_kg,Frailty,Height_m,Weight_kg,BMI,AgeGroup,Frailty_binary,AgeGroup_<30,AgeGroup_30–45,AgeGroup_46–60,AgeGroup_>60
0,65.8,112,30,30,N,1.67132,50.802345,18.19,30–45,0,0,1,0,0
1,71.5,136,19,31,N,1.81610,61.688562,18.70,<30,0,1,0,0,0
2,69.4,153,45,29,N,1.76276,69.399633,22.33,30–45,0,0,1,0,0
3,68.2,142,22,28,Y,1.73228,64.410117,21.46,<30,1,1,0,0,0
4,67.8,144,29,24,Y,1.72212,65.317301,22.02,<30,1,1,0,0,0
5,68.7,123,50,26,N,1.74498,55.791862,18.32,46–60,0,0,0,1,0
6,69.8,141,51,22,Y,1.77292,63.956524,20.35,46–60,1,0,0,1,0
7,70.1,136,23,20,Y,1.78054,61.688562,19.46,<30,1,1,0,0,0
8,67.9,112,17,19,N,1.72466,50.802345,17.08,<30,0,1,0,0,0
9,66.8,120,39,31,N,1.69672,54.431084,18.91,30–45,0,0,1,0,0


# BMI 

In [4]:
# Analyze: create the summary table, quantify the grip/frailty relationship, and save findings.
results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)

numeric_columns = data.select_dtypes(include="number").columns
summary = data[numeric_columns].agg(["mean", "median", "std"]).T
summary.to_csv(results_dir / "numeric_summary.csv")

correlation = data["Grip_kg"].corr(data["Frailty_binary"])
summary_table_lines = [
    "| Column | Mean | Median | Std |",
    "| --- | ---: | ---: | ---: |",
]
for column, values in summary.round(4).iterrows():
    summary_table_lines.append(
        f"| `{column}` | {values['mean']:.4f} | {values['median']:.4f} | {values['std']:.4f} |"
    )

report_lines = [
    "# Frailty Analysis Findings",
    "",
    "## Numeric Summary",
    "",
    "The table below reports the mean, median, and sample standard deviation for every numeric column in the processed dataset.",
    "",
    *summary_table_lines,
    "",
    "## Grip Strength and Frailty",
    "",
    f"The Pearson correlation between `Grip_kg` and `Frailty_binary` is **{correlation:.4f}**.",
    "",
    "A negative value indicates that lower grip strength is associated with frailty in this sample.",
]
(results_dir / "findings.md").write_text("\n".join(report_lines), encoding="utf-8")

print(f"Grip/frailty correlation: {correlation:.4f}")
print(summary.round(2))
print("Saved results/findings.md and results/numeric_summary.csv")

Grip/frailty correlation: -0.4759
                  mean  median    std
Height_in        68.60   68.45   1.67
Weight_lb       131.90  136.00  14.23
Age_yr           32.50   29.50  12.86
Grip_kg          26.00   27.00   4.52
Height_m          1.74    1.74   0.04
Weight_kg        59.83   61.69   6.46
BMI              19.68   19.19   1.78
Frailty_binary    0.40    0.00   0.52
AgeGroup_<30      0.50    0.50   0.53
AgeGroup_30–45    0.30    0.00   0.48
AgeGroup_46–60    0.20    0.00   0.42
AgeGroup_>60      0.00    0.00   0.00
Saved results/findings.md and results/numeric_summary.csv
